In [1]:
import math
import sys
import yaml
import numpy as np
sys.path.append('../../python/')  
from periphery import logicGate
from periphery import constant
from periphery.Technology import Technology
from periphery.sramWriteDriver import SRAMWriteDriver
from periphery.precharger import Precharger
from periphery.WLdecoder import RowDecoder
from periphery.SenseAmp import SenseAmp
from periphery.DFF import DFF
from periphery.MUX import Mux
from periphery.levelShifter import LevelShifter
from periphery.WLDecoderDriver import WLNewDecoderDriver
from periphery.adder import Adder
from periphery.ADC import SarADC
print(constant.INV)

0
0


In [3]:
with open('../../config.yaml', 'r') as file:
    config = yaml.safe_load(file)

with open('../../mapping.yaml', 'r') as file:
    mapping = yaml.safe_load(file)

with open('../../param.yaml', 'r') as file:
    param = yaml.safe_load(file)

with open('../../RNG.yaml', 'r') as file:
    RNG = yaml.safe_load(file)

In [64]:
class SubArray:
    def __init__(self, tech, param, config, mapping, RNG, numRow, numCol, relaxArrayCellWidth = False,relaxArrayCellHeight = False):
        self.tech = tech
        self.param = param  
        self.config = config 
        self.mapping = mapping
        self.RNG = RNG
        self.relaxArrayCellWidth = relaxArrayCellWidth
        self.relaxArrayCellHeight = relaxArrayCellHeight

        self.numRow = numRow
        self.numCol = numCol

        self.precision_sigma = self.config['precision_sigma']
        self.precision_ADC = self.config['precision_ADC']
        self.clk_freq = self.config['frequency']

        self.SubArray_sigma = SubArray_sigma(numCol=self.numCol,numRow = self.numRow,relaxArrayCellWidth = False,relaxArrayCellHeight = False,tech=self.tech,config=self.config,mapping=self.mapping,param=self.param,RNG=self.RNG)
        self.SubArray_mu = SubArray_mu(numCol=self.numCol,numRow = self.numRow,tech=self.tech,config=self.config,mapping=self.mapping,param=self.param)
        #if mu is sram, seed the shift register to extend the bitline to match witht the sigma array
        self.Adder = Adder(num_bit=self.precision_ADC,num_adder=self.numCol,clk_freq = None,tech=self.tech,config=self.config,mapping=self.mapping) 
        self.dff = DFF(num_dff=(self.precision_ADC + 1) *self.numCol,tech=self.tech,config=self.config,mapping=self.mapping,param=self.param,clk_freq=self.clk_freq)

    def calculate_area(self):
        if not self.initialized:
            raise Exception("SubArray not initialized. Please call initialize() method first.")
        else:
            area = 0
            used_area = 0
            Adder_area,Adder_height,Adder_width = self.Adder.calculate_area(new_height=None,new_width=None,option='NONE')
            dff_area,dff_height,dff_width = self.dff.calculate_area(new_height=None,new_width=None,option='NONE')
            SubArray_mu_area,SubArray_mu_height,SubArray_mu_width,SubArray_mu_used_area = self.SubArray_mu.calculate_area()
            SubArray_sigma_area,SubArray_sigma_height,SubArray_sigma_width,SubArray_sigma_used_area = self.SubArray_sigma.calculate_area()

            height = max(SubArray_mu_height,SubArray_sigma_height) + dff_height + Adder_height
            width = SubArray_mu_width + SubArray_sigma_width

            area = height * width

        return area, height, width
    
    def calculate_latency(self, column_res, calculate_clk_freq, validated=False):
        if not self.initialized:
            raise Exception("SubArray not initialized. Please call initialize() method first.")
        else:
            read_latency = 0
            read_latency_adc = 0
            read_latency_other = 0
            write_latency = 0
            SubArray_sigma_read_latency = self.SubArray_sigma.calculate_latency(calculate_clk_freq = self.clk_freq,validated=False)
            SubArray_mu_read_latency = self.SubArray_mu.calculate_latency(calculate_clk_freq = self.clk_freq,validated=False)
            Adder_read_latency = self.Adder.calculate_latency(cap_load=0,num_read=1)
            dff_read_latency,dff_write_latency = self.dff.calculate_latency(num_read=1)
            read_latency = max(SubArray_sigma_read_latency,SubArray_mu_read_latency) + Adder_read_latency + dff_read_latency
            if validated:
                read_latency *= self.param['beta']

        return read_latency

    def calculate_power(self, input_vector, weight_matrix):
        if not self.initialized:
            raise Exception("SubArray not initialized. Please call initialize() method first.")
        else:
            readDynamicEnergy = 0
            write_dynamic_energy = 0
            read_dynamic_energy_array = 0

            #######################################################################
            weight_matrix = np.load('../../slice.npy')
            num_rows = weight_matrix.shape[0]
            input_vector = [1] * num_rows
            #######################################################################
            SubArray_mu_read_energy,SubArray_mu_write_energy,SubArray_mu_leakage = self.SubArray_mu.calculate_power(input_vector, weight_matrix)
            SubArray_sigma_read_energy,SubArray_sigma_write_energy,SubArray_sigma_leakage = self.SubArray_sigma.calculate_power(input_vector, weight_matrix)
            Adder_read_energy,Adder_leakage = self.Adder.calculate_power(num_read=1,num_adder_per_op=1)
            dff_read_latency,dff_write_latency = self.dff.calculate_latency(num_read=1)


            leakage = SubArray_mu_leakage + SubArray_sigma_leakage + Adder_leakage
            read_dynamic_energy_array = SubArray_mu_read_energy + SubArray_sigma_read_energy
            readDynamicEnergy = read_dynamic_energy_array + Adder_read_energy
            write_dynamic_energy = SubArray_mu_write_energy + SubArray_sigma_write_energy
            

        return readDynamicEnergy, write_dynamic_energy, leakage
            

In [65]:
tech45 = Technology(node_nm=65, roadmap='HP')
# Instantiate and initialize Precharger
pre = SubArray_mu(
    numCol=8*8,
    numRow = 64,
    tech=tech45,
    config=config,
    mapping=mapping,
    param=param
)

In [66]:
area,height,width,used_area = pre.calculate_area()


print("Width Result:", width)
print("Height Result:", height)
print("Area Result:", area*1e6)
print("Used Area Result:", used_area*1e6)

width_array 0.00011648
height_array 4.16e-05
Width Result: 0.000127764
Height Result: 5.1194e-05
Area Result: 0.006540750216
Used Area Result: 0.006767441407999999


In [67]:
column_resistance_list = [1e6] #* 1_000_000
read_latency = pre.calculate_latency(
    column_res=None,
    calculate_clk_freq = 1e6,
    validated=False
)
print("Read Latency:", read_latency)

wl_decoder_read_latency 2.8575499144358534e-10
precharger_read_latency 8.991784145110508e-11
col_delay 3.8903255694576576e-11
sense_amp_read_latency 2.927225657322598e-12
Read Latency: 9.175033142465896e-10


In [68]:
#######################################################################
weight_matrix = np.load('../../slice.npy')
num_rows = weight_matrix.shape[0]
input_vector = [1] * num_rows
#######################################################################
read_energy,write_energy,leakage = pre.calculate_power(input_vector, weight_matrix)

print(f"  Read Dynamic Energy: {read_energy:.3e} J")
print(f"  Write Dynamic Energy: {write_energy:.3e} J")
print(f"  Leakage Power: {leakage:.3e} W")

  Read Dynamic Energy: 1.107e-10 J
  Write Dynamic Energy: 0.000e+00 J
  Leakage Power: 1.329e-04 W
